In [44]:
!pip install torch

In [45]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import glob
import os
import unicodedata
import string
import random

In [46]:
!wget https://download.pytorch.org/tutorial/data.zip

--2026-03-15 14:14:35--  https://download.pytorch.org/tutorial/data.zip
Resolving download.pytorch.org (download.pytorch.org)... 18.160.10.22, 18.160.10.28, 18.160.10.76, ...
Connecting to download.pytorch.org (download.pytorch.org)|18.160.10.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2882130 (2.7M) [application/zip]
Saving to: ‘data.zip.4’

data.zip.4          100%[===================>]   2.75M  --.-KB/s    in 0.01s   

2026-03-15 14:14:35 (206 MB/s) - ‘data.zip.4’ saved [2882130/2882130]



In [47]:
!unzip -o data.zip

Archive:  data.zip
  inflating: data/eng-fra.txt        
  inflating: data/names/Arabic.txt   
  inflating: data/names/Chinese.txt  
  inflating: data/names/Czech.txt    
  inflating: data/names/Dutch.txt    
  inflating: data/names/English.txt  
  inflating: data/names/French.txt   
  inflating: data/names/German.txt   
  inflating: data/names/Greek.txt    
  inflating: data/names/Irish.txt    
  inflating: data/names/Italian.txt  
  inflating: data/names/Japanese.txt  
  inflating: data/names/Korean.txt   
  inflating: data/names/Polish.txt   
  inflating: data/names/Portuguese.txt  
  inflating: data/names/Russian.txt  
  inflating: data/names/Scottish.txt  
  inflating: data/names/Spanish.txt  
  inflating: data/names/Vietnamese.txt  


In [48]:
all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)

print("Number of characters:", n_letters)

Number of characters: 57


In [49]:
def unicodeToAscii(s):

    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

In [50]:
print(unicodeToAscii("Ślusàrski"))

Slusarski


In [51]:
def findFiles(path):
    return glob.glob(path)

print(findFiles('data/names/*.txt'))

['data/names/English.txt', 'data/names/Korean.txt', 'data/names/Italian.txt', 'data/names/Arabic.txt', 'data/names/Russian.txt', 'data/names/French.txt', 'data/names/Irish.txt', 'data/names/Spanish.txt', 'data/names/Vietnamese.txt', 'data/names/Portuguese.txt', 'data/names/Dutch.txt', 'data/names/German.txt', 'data/names/Polish.txt', 'data/names/Scottish.txt', 'data/names/Chinese.txt', 'data/names/Greek.txt', 'data/names/Czech.txt', 'data/names/Japanese.txt']


In [52]:
category_lines = {}
all_categories = []

for filename in findFiles('data/names/*.txt'):

    category = os.path.splitext(os.path.basename(filename))[0]
    all_categories.append(category)

    lines = open(filename, encoding='utf-8').read().strip().split('\n')

    lines = [unicodeToAscii(line) for line in lines]

    category_lines[category] = lines

n_categories = len(all_categories)

print("Languages:", all_categories)
print("Number of languages:", n_categories)

Languages: ['English', 'Korean', 'Italian', 'Arabic', 'Russian', 'French', 'Irish', 'Spanish', 'Vietnamese', 'Portuguese', 'Dutch', 'German', 'Polish', 'Scottish', 'Chinese', 'Greek', 'Czech', 'Japanese']
Number of languages: 18


In [53]:
def letterToIndex(letter):
    return all_letters.find(letter)

In [54]:
def letterToTensor(letter):

    tensor = torch.zeros(1, n_letters)

    tensor[0][letterToIndex(letter)] = 1

    return tensor

In [55]:
print(letterToTensor('A'))

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.]])


In [56]:
def lineToTensor(line):

    tensor = torch.zeros(len(line), n_letters)

    for li, letter in enumerate(line):
        tensor[li][letterToIndex(letter)] = 1

    return tensor

In [57]:
print(lineToTensor("Lee").size())

torch.Size([3, 57])


In [58]:
class RNNClassifier(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):

        super(RNNClassifier, self).__init__()

        self.hidden_size = hidden_size

        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size)

        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):

        out, hidden = self.rnn(x)

        out = self.fc(hidden[-1])

        out = self.softmax(out)

        return out

In [59]:
hidden_size = 256

model = RNNClassifier(n_letters, hidden_size, n_categories)

print(model)

RNNClassifier(
  (rnn): RNN(57, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=18, bias=True)
  (softmax): LogSoftmax(dim=1)
)


In [60]:
criterion = nn.NLLLoss()

optimizer = optim.Adam(model.parameters(), lr=0.002
                       )

In [61]:
def randomTrainingExample():

    category = random.choice(all_categories)

    line = random.choice(category_lines[category])

    category_tensor = torch.tensor([all_categories.index(category)])

    line_tensor = lineToTensor(line)

    return category, line, category_tensor, line_tensor

In [62]:
def train():

    category, line, category_tensor, line_tensor = randomTrainingExample()

    optimizer.zero_grad()

    output = model(line_tensor.unsqueeze(0))

    loss = criterion(output, category_tensor)

    loss.backward()

    optimizer.step()

    return loss.item(), line, category

In [63]:
for i in range(150000):

    loss, line, category = train()

    if i % 5000 == 0:
        print(i, loss, line, category)

0 2.855239152908325 Skala Polish
5000 4.0339579582214355 Ahn Korean
10000 0.910531222820282 Bei Chinese
15000 2.4173266887664795 Wan Chinese
20000 1.364822506904602 Martinez Spanish
25000 3.8336598873138428 Rocha Portuguese
30000 1.9036791324615479 Leeuwenhoek Dutch
35000 0.19693075120449066 Wan Chinese
40000 3.355635166168213 Honjas Greek
45000 0.029225626960396767 Cui Chinese
50000 5.081286430358887 Chavez Spanish
55000 2.831605911254883 Murray Scottish
60000 3.322054147720337 Ramos Spanish
65000 3.0905308723449707 Winward English
70000 0.3718283772468567 Min Chinese
75000 4.030291557312012 Ashida Japanese
80000 2.3221328258514404 Karubo Japanese
85000 5.562779903411865 Yoshinobu Japanese
90000 3.2922210693359375 Neisser Czech
95000 0.0014228230575099587 Si Korean
100000 3.016509771347046 Lim Chinese
105000 1.9377061128616333 Gorchak Russian
110000 1.1868047714233398 Lamar French
115000 0.5733575224876404 Dao Vietnamese
120000 4.63739538192749 Campbell Scottish
125000 2.0768468379974

In [64]:
def predict(name):

    with torch.no_grad():

        tensor = lineToTensor(name)

        output = model(tensor.unsqueeze(0))

        topv, topi = output.topk(1)

        category = all_categories[topi.item()]

        print(name, "→", category)

In [65]:
predict("Ivanov")
predict("Schmidt")
predict("Garcia")
predict("Kim")
predict("Dubois")

Ivanov → Russian
Schmidt → Russian
Garcia → Spanish
Kim → Chinese
Dubois → Spanish
